# 泰坦尼克号生存预测

> 二分类 · 表格数据 · 特征工程入门

用乘客的舱位、性别、年龄、票价等信息，预测其能否在泰坦尼克号沉船中生还。
数据来自 Kaggle [Titanic - Machine Learning from Disaster](https://www.kaggle.com/c/titanic)，共 891 条训练样本。

**目录**

1. [数据概览](#1-数据概览)
2. [数据清洗](#2-数据清洗)
3. [特征工程](#3-特征工程)
4. [建模与评估](#4-建模与评估)
5. [结论](#5-结论)

## 1. 数据概览

先看数据长什么样、缺什么、目标变量怎么分布。

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

# 优先读同目录下的 data.csv（从 Kaggle 下载后改名）；
# 没有本地文件时回退到在线数据源，方便在云端 Jupyter 里直接运行。
DATA_PATH = Path("data.csv")
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    )

df.head()

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  

In [2]:
print("样本数:", df.shape[0], "| 字段数:", df.shape[1])
print()
df.info()

样本数: 891 | 字段数: 12

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [3]:
# 每列缺多少
df.isnull().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

三列有缺失，处理方式各不相同：

| 字段 | 缺失数 | 处理 |
| --- | --- | --- |
| `Cabin` | 687（77%） | 直接删除，缺失太多，强行填充只会引入噪声 |
| `Age` | 177（20%） | 中位数填充，分布右偏，中位数比均值更稳 |
| `Embarked` | 2 | 这两条整行删除，样本量小，不影响结论 |

In [4]:
print(df["Survived"].value_counts())
print()
print("整体生还率: {:.1%}".format(df["Survived"].mean()))

Survived
0    549
1    342
Name: count, dtype: int64

整体生还率: 38.4%


### 哪些特征一眼就能看出有用

按性别、舱位分组看生还率差异，差异越大，特征的区分力越强。

In [5]:
print("按性别:")
print(df.groupby("Sex")["Survived"].agg(["mean", "count"]).round(3))
print()
print("按舱位:")
print(df.groupby("Pclass")["Survived"].agg(["mean", "count"]).round(3))

按性别:
         mean  count
Sex                 
female  0.742    314
male    0.189    577

按舱位:
         mean  count
Pclass              
1       0.630    216
2       0.473    184
3       0.242    491


**结论**：女性生还率约 74%，男性约 19%；头等舱约 63%，三等舱约 24%。
性别和舱位是最强的两个信号，3.1 会用「只用性别」的决策树验证这一点。

## 2. 数据清洗

清洗和特征提取分两步走：先从 `Name` 里抠出称呼（`Title`），再把用不上的原始文本列删掉。

In [6]:
# 从 "Braund, Mr. Owen Harris" 里提取 "Mr"
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.")
print(df["Title"].value_counts())

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


称呼里有一堆低频类别（Dr、Rev、Col、Countess……），样本太少会让模型过拟合，统一归为 `Rare`。

In [7]:
# 出现次数少于 10 次的称呼合并成 Rare
rare_titles = df["Title"].value_counts()[df["Title"].value_counts() < 10].index
df["Title"] = df["Title"].replace(rare_titles, "Rare")
print(df["Title"].value_counts())

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64


In [8]:
# 类别编码：Mr=0, Miss=1, Mrs=2, Master=3, Rare=4
df["Title"] = df["Title"].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})

# 删掉用不上的列（Name 已提取成 Title，不再需要）
df = df.drop(columns=["Name", "Cabin", "Ticket", "PassengerId"])

# 缺失值处理
df = df.dropna(subset=["Embarked"])          # 只缺 2 条
df["Age"] = df["Age"].fillna(df["Age"].median())

# 类别编码
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

print("清洗后形状:", df.shape)
print("剩余缺失值:", int(df.isnull().sum().sum()))
df.head()

清洗后形状: (889, 9)
剩余缺失值: 0


   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  Title
0         0       3    0  22.0      1      0   7.2500         0      0
1         1       1    1  38.0      1      0  71.2833         1      2
2         1       3    1  26.0      0      0   7.9250         0      1
3         1       1    1  35.0      1      0  53.1000         0      2
4         0       3    0  35.0      0      0   8.0500         0      0

## 3. 建模与评估

按 8:2 划分训练集和测试集，`stratify=y` 保证两边生还比例一致。

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"训练集 {len(X_train)} 条，测试集 {len(X_test)} 条")
print("特征:", list(X.columns))

训练集 711 条，测试集 178 条
特征: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title']


### 3.1 基线：只用性别

在动用所有特征之前，先用一个特征算出下限，后面每个模型的提升才有参照。

In [10]:
from sklearn.tree import DecisionTreeClassifier

X_sex = df[["Sex"]]
X_tr, X_te, y_tr, y_te = train_test_split(X_sex, y, test_size=0.2, random_state=42, stratify=y)

sex_tree = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
print(f"只用 Sex 一个特征: {sex_tree.score(X_te, y_te):.2%}")

只用 Sex 一个特征: 76.97%


### 3.2 决策树

换回全部特征。默认的决策树会一直分裂到每个叶子纯净，先看它，再限制深度对比。

In [11]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
print(f"决策树（默认深度）准确率: {tree.score(X_test, y_test):.2%}")

决策树（默认深度）准确率: 76.40%


In [12]:
# 限制树的深度，防止过拟合
for depth in [3, 5, 7, 10]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    print(f"max_depth={depth:<2} 准确率: {m.score(X_test, y_test):.2%}")

max_depth=3  准确率: 83.15%
max_depth=5  准确率: 82.58%
max_depth=7  准确率: 80.90%
max_depth=10 准确率: 78.09%


深度从 3 一路加到 10，准确率反而往下掉——树越深，越容易把训练集里的噪声也背下来，这就是过拟合。
这里深度 3 最好，说明信号本来就集中在少数几个明显的分界上（性别、舱位、称呼）。

### 3.3 加入 Title 有用吗（消融实验）

固定其他条件，只增删 `Title` 这一列，看准确率有没有变化。

In [13]:
base = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

for label, cols in [("基础特征", base), ("基础特征 + Title", base + ["Title"])]:
    Xa, Xb, ya, yb = train_test_split(df[cols], y, test_size=0.2, random_state=42, stratify=y)
    m = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xa, ya)
    print(f"{label}: {m.score(Xb, yb):.2%}")

基础特征: 81.46%
基础特征 + Title: 83.15%


### 3.4 随机森林

用 300 棵树投票，比单棵决策树更稳，也不容易过拟合。

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
print(f"随机森林准确率: {rf.score(X_test, y_test):.2%}")

随机森林准确率: 80.90%


### 3.5 KNN（必须标准化）

KNN 按距离算：`Fare` 能到 500 多，而 `SibSp` 只有 0~8，不标准化时票价会完全主导距离。
第一版代码没做标准化，KNN 只有 73%；套一层 `StandardScaler` 之后，才能和树模型放在一起比。

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

for k in [3, 5, 7, 9]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(X_train, y_train)
    print(f"KNN（标准化后）k={k} 准确率: {knn.score(X_test, y_test):.2%}")

KNN（标准化后）k=3 准确率: 79.21%
KNN（标准化后）k=5 准确率: 79.21%
KNN（标准化后）k=7 准确率: 78.65%
KNN（标准化后）k=9 准确率: 80.90%


### 3.6 混淆矩阵

准确率看不出错在哪儿，混淆矩阵能。

In [16]:
from sklearn.metrics import confusion_matrix, classification_report

pred = rf.predict(X_test)
print(confusion_matrix(y_test, pred))
print()
print(classification_report(y_test, pred, target_names=["遇难", "生还"]))

[[94 16]
 [18 50]]

              precision    recall  f1-score   support

          遇难       0.84      0.85      0.85       110
          生还       0.76      0.74      0.75        68

    accuracy                           0.81       178
   macro avg       0.80      0.79      0.80       178
weighted avg       0.81      0.81      0.81       178



### 3.7 交叉验证

单次划分受随机性影响大，5 折交叉验证更能反映真实水平。

In [17]:
from sklearn.model_selection import cross_val_score

models = {
    "决策树 (max_depth=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "随机森林 (300 棵)": RandomForestClassifier(n_estimators=300, random_state=42),
}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)
    print(f"{name}: 每次 {scores.round(4)}")
    print(f"{'':<20} 平均 {scores.mean():.2%}")

决策树 (max_depth=3): 每次 [0.8427 0.809  0.8202 0.7809 0.8475]
                     平均 82.01%
随机森林 (300 棵): 每次 [0.7753 0.7978 0.8652 0.7584 0.8023]
                     平均 79.98%


## 4. 结论

| 模型 | 测试集准确率 | 5 折交叉验证 |
| --- | --- | --- |
| 只用性别（基线） | 76.97% | — |
| 决策树，默认深度 | 76.40% | — |
| 决策树，max_depth=3 | **83.15%** | **82.01%** |
| 随机森林，300 棵 | 80.90% | 79.98% |
| KNN，标准化后 | 78.65% ~ 80.90% | — |

**三个值得写进简历的发现**

1. **性别一个特征就值 77%**：只用 `Sex` 训练决策树，测试集准确率 76.97%，而完整特征调优后是 83.15%——七列特征加起来只多出 6 个百分点，说明这个数据集的信号高度集中。
2. **`Title` 是这轮特征工程里唯一有效的**：把 "Mr / Mrs / Miss / Master" 从姓名里拆出来，准确率从 81.46% 提到 83.15%，涨了 1.7 个百分点。它之所以有用，是因为能区分"成年男性"和"小男孩"，而单纯的性别做不到。
3. **树不是越深越好，模型也不是越复杂越好**：深度从 3 加到 10，准确率从 83.15% 掉到 78.09%；300 棵树的随机森林交叉验证只有 79.98%，反而不如一棵限制深度的树。数据量小、信号简单的时候，简单模型更抗过拟合。

**踩过的坑**

- KNN 一开始只有 73%，不是模型不行，是没做标准化——`Fare` 的量级（0~512）完全压过了 `SibSp`（0~8），距离几乎只由票价决定。加了 `StandardScaler` 之后回到 78% 以上。
- 中间有一版把 `df.head` 写成了不带括号的引用，输出的是方法对象而不是数据表，排查时白费了时间。

**可以继续做**

1. 用 `GridSearchCV` 在 `max_depth`、`min_samples_leaf`、`min_samples_split` 上细调这棵树；
2. 构造 `FamilySize = SibSp + Parch + 1` 和 `IsAlone`，看"独自一人"是否影响生还；
3. 用 `cross_val_score(scoring="f1")` 再评估一遍——准确率会掩盖"生还"这一类判得差的问题；
4. 换 XGBoost / LightGBM 做对比，看提升是否值得复杂度。